# 1. Research Question

How is trader profitability associated with (1) market breadth and (2) concentration of participation? In particular, do broad but concentrated traders differ from broad and evenly distributed traders?

The supplied file contains aggregate trader-level features. It supports **`markets_per_day` as a market-breadth measure**, not a lifetime count of distinct markets. It does not identify how transactions are allocated across individual markets, so the exact market-concentration question cannot be answered. A separate, explicitly labeled topic-concentration proxy is explored below.

The primary outcome is `trader_ppv` (profit per unit of trading volume). `trader_pnl` (total profit/loss) is secondary. Each trader has equal weight. Associations are not causal effects.

# 2. Data Loading

This notebook runs independently from the folder containing `data_no_imputation.parquet`. Dependencies: pandas, NumPy, Matplotlib, SciPy, IPython, and PyArrow. It only reads the source Parquet file.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown
import pyarrow.parquet as pq

plt.rcParams.update({"figure.figsize": (11, 6), "font.size": 11})
pd.set_option("display.max_columns", 20)

df = pd.read_parquet("data_no_imputation.parquet")
print(f"Dataset shape: {df.shape[0]:,} rows ? {df.shape[1]:,} columns")
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing_values": df.isna().sum()}))
print("Available columns:", df.columns.tolist())

breadth_column = "markets_per_day"
topic_proxy_column = "largest_transformers_topic_share"
required_columns = ["trader", "transaction_count", breadth_column, "trader_ppv", "trader_pnl"]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
print("All required core columns exist.")

# Inspect the complete schema rather than inferring market data from a name alone.
market_candidates = [c for c in df if any(word in c.lower() for word in ["market", "condition_id", "token_id"])]
print("Market-related columns:", market_candidates)
print("Arrow schema:")
print(pq.read_schema("data_no_imputation.parquet"))
print("This supplied schema contains only markets_per_day for markets: no market identifiers, per-market count maps, or transaction-level rows.")
print("Distinct market counts and effective market counts are unavailable.")

# 3. Data Preparation

Verify unique trader identifiers; retain traders with at least 100 transactions and finite values for breadth, PPV, and P&L. Preserve extreme finite observations. Negative breadth is a data-quality error; zeros are retained.

The optional topic analysis uses an explicitly reported subset with finite, valid topic-proxy values. Missing topic information does not remove traders from the breadth-only analysis. Topic-share diagnostics use the full loaded dataset and do not impute or renormalize values.

In [ ]:
ids = df["trader"].astype("string").str.strip()
missing_ids = ids.isna() | ids.eq("").fillna(False)
duplicate_ids = ids.loc[~missing_ids].duplicated(keep=False)
print(f"Missing/blank trader IDs: {missing_ids.sum():,}; rows with repeated IDs: {duplicate_ids.sum():,}")
if missing_ids.any() or duplicate_ids.any():
    raise ValueError("Resolve trader identifiers before treating rows as individual traders.")

numeric_columns = ["transaction_count", breadth_column, "trader_ppv", "trader_pnl"]
analysis_df = df[["trader", *numeric_columns]].copy()
for column in numeric_columns:
    analysis_df[column] = pd.to_numeric(analysis_df[column], errors="raise")
analysis_df = analysis_df.loc[analysis_df["transaction_count"] >= 100].copy()
threshold_count = len(analysis_df)
finite_rows = np.isfinite(analysis_df[numeric_columns].to_numpy(dtype=float, na_value=np.nan)).all(axis=1)
analysis_df = analysis_df.loc[finite_rows].copy()
print(f"Traders meeting the transaction threshold: {threshold_count:,}")
print(f"Removed for missing/infinite relevant values: {threshold_count - len(analysis_df):,}")
print(f"Traders remaining for breadth analysis: {len(analysis_df):,}")
if len(analysis_df) < 3:
    raise ValueError("At least three valid traders are needed.")
if (analysis_df[breadth_column] < 0).any():
    raise ValueError("Negative markets_per_day found; inspect source values.")
if analysis_df[breadth_column].nunique() < 2:
    raise ValueError("Market breadth has no variation.")
display(analysis_df[numeric_columns].describe(percentiles=[0.01, 0.5, 0.9, 0.99]))

# 4. Diversification Metric Construction

### Individual-market diversification is unavailable

`markets_per_day` measures a daily rate of breadth. It is **not the total number of distinct markets**, and multiplying it by an undocumented activity window would not recover that count because a trader can revisit a market.

To calculate distinct markets and $N_{effective}=1/\sum_i p_i^2$, we need either transaction-level **trader ID, stable market ID, transaction ID/count, and timestamps defining a common observation window**, or per-trader/per-market transaction counts for that same window. Deduplicate transaction records and specify how fills, buys, and sells count. Then $p_i=n_i/\sum_j n_j$, distinct markets count positive $n_i$, and effective markets use those shares. Multiple outcome tokens must map to their parent prediction market.

### Topic diagnostics and a separate existing proxy

Audit `topic_*` fields for numeric ranges, finite values, row totals, and agreement of their maximum with `largest_transformers_topic_share`. A common [0,1] scale and totals near one establish numerical normalization, **not** mutually exclusive topic assignment or a transaction-count denominator. No topic-generation definitions or overlap documentation accompany this file; overlap remains unknown.

Therefore, do not construct a topic HHI, entropy, effective topic count, or effective market count. Use the **existing** largest transformer-topic share as a descriptive **topic-concentration proxy** only when the numerical checks pass. Higher values indicate a more dominant topic in the supplied representation. Lower values do not prove evenly distributed activity across markets. Do not combine the transformer and tag share fields: their taxonomies and denominators are undocumented.

In [ ]:
topic_columns = [c for c in df.columns if c.startswith("topic_")]
topic_proxy_available = False
topic_audit = pd.DataFrame()
if topic_columns and topic_proxy_column in df.columns:
    topic_values = df[topic_columns].apply(pd.to_numeric, errors="raise")
    topic_array = topic_values.to_numpy(dtype=float, na_value=np.nan)
    complete = np.isfinite(topic_array).all(axis=1)
    complete_values = topic_array[complete]
    row_sums = complete_values.sum(axis=1)
    bounded = ((complete_values >= 0) & (complete_values <= 1)).all(axis=1)
    normalized = np.isclose(row_sums, 1.0, rtol=1e-6, atol=1e-8)
    proxy_values = pd.to_numeric(df[topic_proxy_column], errors="raise").to_numpy(dtype=float, na_value=np.nan)[complete]
    proxy_finite = np.isfinite(proxy_values)
    comparable = proxy_finite & bounded & normalized
    matches = np.isclose(complete_values.max(axis=1), proxy_values, rtol=1e-6, atol=1e-8)
    topic_audit = pd.DataFrame([{
        "topic_columns": len(topic_columns), "complete_topic_rows": int(complete.sum()),
        "incomplete_topic_rows": int((~complete).sum()),
        "bounded_rows": int(bounded.sum()), "normalized_rows": int(normalized.sum()),
        "comparable_rows": int(comparable.sum()),
        "max_matches_existing_proxy": int((matches & comparable).sum()),
        "min_row_sum": row_sums.min() if row_sums.size else np.nan,
        "max_row_sum": row_sums.max() if row_sums.size else np.nan,
    }])
    display(topic_audit)
    topic_proxy_available = bool(comparable.any() and bounded.all() and normalized.all() and matches[comparable].all())
    print("Numerical proxy checks passed:", topic_proxy_available)
else:
    print("Topic fields needed for proxy validation are unavailable.")
print("Underlying topic overlap and weighting denominator: undocumented; not established by normalization.")
print("No concentration index is calculated from the topic columns.")

topic_df = pd.DataFrame()
if topic_proxy_available:
    topic_df = analysis_df.copy()
    topic_df[topic_proxy_column] = pd.to_numeric(df.loc[topic_df.index, topic_proxy_column], errors="raise")
    finite_topic = np.isfinite(topic_df[topic_proxy_column].to_numpy(dtype=float, na_value=np.nan))
    topic_df = topic_df.loc[finite_topic].copy()
    if not topic_df[topic_proxy_column].between(0, 1).all():
        raise ValueError("Existing topic share is outside [0,1].")
    print(f"Traders in topic-proxy analysis: {len(topic_df):,}; removed for missing/infinite proxy: {len(analysis_df)-len(topic_df):,}")
    topic_proxy_available = not topic_df.empty
else:
    print("Topic heatmaps will be skipped: a numerically validated proxy is unavailable.")

In [ ]:
def quantile_groups(values, requested=10):
    if values.nunique() < 2:
        return pd.Series(pd.Categorical(["All traders"] * len(values)), index=values.index)
    return pd.qcut(values, q=requested, duplicates="drop")

def profitability_summary(data, group_column, metric):
    return data.groupby(group_column, observed=True).agg(
        n_traders=("trader_ppv", "size"),
        min_metric=(metric, "min"), median_metric=(metric, "median"), max_metric=(metric, "max"),
        mean_trader_ppv=("trader_ppv", "mean"), median_trader_ppv=("trader_ppv", "median"),
        mean_trader_pnl=("trader_pnl", "mean"), median_trader_pnl=("trader_pnl", "median"),
    ).reset_index()

analysis_df["breadth_group"] = quantile_groups(analysis_df[breadth_column])
breadth_summary = profitability_summary(analysis_df, "breadth_group", breadth_column)
breadth_summary.insert(0, "group", [f"B{i+1}" for i in range(len(breadth_summary))])
assert breadth_summary["n_traders"].sum() == len(analysis_df)
print(f"Requested 10 breadth quantiles; obtained {len(breadth_summary)} nonempty groups.")
print("Tied values stay together; repeated boundaries can reduce group count and make counts unequal.")
display(breadth_summary.round(6))

if topic_proxy_available:
    topic_df["topic_group"] = quantile_groups(topic_df[topic_proxy_column])
    topic_summary = profitability_summary(topic_df, "topic_group", topic_proxy_column)
    print("Separate TOPIC-concentration proxy summary (not individual-market concentration):")
    display(topic_summary.round(6))

# 5. Visualizations

All profitability Y-axes and heatmap profit values use **`trader_ppv`**, not total P&L. Group means and medians weight each trader equally. All finite extremes are retained. Breadth uses a log X-axis only when its positive range spans at least two orders of magnitude; a symmetric-log axis preserves zeros if needed.

### Chart 1: Market breadth versus profitability

In [ ]:
breadth_values = analysis_df[breadth_column].to_numpy(dtype=float)
positive = breadth_values[breadth_values > 0]
span = positive.max() / positive.min() if positive.size else 1.0
breadth_scale = ("symlog" if (breadth_values == 0).any() else "log") if span >= 100 else "linear"
print(f"Positive breadth max/min: {span:,.1f}; selected X-axis scale: {breadth_scale}")

def format_breadth_axis(ax):
    if breadth_scale == "symlog":
        ax.set_xscale("symlog", linthresh=float(positive.min()))
    else:
        ax.set_xscale(breadth_scale)
    ax.set_xlabel(f"Markets per day ({breadth_scale} scale; not total distinct markets)")
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.grid(alpha=0.2)

fig, ax = plt.subplots()
ax.scatter(breadth_values, analysis_df["trader_ppv"], s=12, alpha=0.15, edgecolors="none")
ax.set_title("Market breadth vs. profit per volume | traders with 100+ transactions")
ax.set_ylabel("Profit per unit of trading volume (trader_ppv)")
format_breadth_axis(ax)
fig.tight_layout()
plt.show()

### Chart 2: Mean profitability by market breadth

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(breadth_summary["median_metric"], breadth_summary["mean_trader_ppv"], marker="o")
for i, row in breadth_summary.iterrows():
    ax.annotate(
        f"{row['group']}\nn={row['n_traders']:,}",
        (row["median_metric"], row["mean_trader_ppv"]),
        xytext=(0, 12 if i % 2 == 0 else -32), textcoords="offset points",
        ha="center", fontsize=9,
    )
ax.set_title("Mean profitability by market-breadth quantile group")
ax.set_ylabel("Mean trader_ppv (equal trader weights)")
format_breadth_axis(ax)
ax.margins(y=0.3)
fig.tight_layout()
plt.show()

### Chart 3: Median profitability by market breadth

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(breadth_summary["median_metric"], breadth_summary["median_trader_ppv"], marker="o")
for i, row in breadth_summary.iterrows():
    ax.annotate(
        f"{row['group']}\nn={row['n_traders']:,}",
        (row["median_metric"], row["median_trader_ppv"]),
        xytext=(0, 12 if i % 2 == 0 else -32), textcoords="offset points",
        ha="center", fontsize=9,
    )
ax.set_title("Median profitability by market-breadth quantile group")
ax.set_ylabel("Median trader_ppv (equal trader weights)")
format_breadth_axis(ax)
ax.margins(y=0.3)
fig.tight_layout()
plt.show()

### Chart 4: Effective market count versus profitability ? unavailable

No figure is generated: the source file has neither market identifiers nor per-market transaction counts. `markets_per_day` and topic shares cannot recover $p_i$ for individual markets. The additional records required are listed in Section 4. No effective market count is invented.

### Chart 5: Market breadth ? TOPIC concentration proxy

These heatmaps are a **cross-topic proxy analysis**, not an analysis of individual-market participation concentration. X is market-breadth quintile; Y is the existing largest transformer-topic share quintile (higher means greater topic dominance). Quintiles preserve ties and may yield fewer than five groups. The first figure displays median PPV in every nonempty cell; empty cells show ?No data?. A separate count heatmap exposes sparse cells. Profit uses a symmetric color range around zero; count uses a sequential color scale.

In [ ]:
joint_summary = pd.DataFrame()
if topic_proxy_available:
    topic_df["heat_breadth_group"] = quantile_groups(topic_df[breadth_column], requested=5)
    topic_df["heat_topic_group"] = quantile_groups(topic_df[topic_proxy_column], requested=5)
    breadth_levels = topic_df["heat_breadth_group"].cat.categories
    topic_levels = topic_df["heat_topic_group"].cat.categories
    joint_summary = topic_df.groupby(["heat_topic_group", "heat_breadth_group"], observed=False).agg(
        n_traders=("trader_ppv", "size"),
        mean_trader_ppv=("trader_ppv", "mean"), median_trader_ppv=("trader_ppv", "median"),
        mean_trader_pnl=("trader_pnl", "mean"), median_trader_pnl=("trader_pnl", "median"),
    ).reset_index()
    median_grid = joint_summary.pivot(index="heat_topic_group", columns="heat_breadth_group", values="median_trader_ppv").reindex(index=topic_levels, columns=breadth_levels)
    count_grid = joint_summary.pivot(index="heat_topic_group", columns="heat_breadth_group", values="n_traders").reindex(index=topic_levels, columns=breadth_levels).fillna(0).astype(int)
    assert count_grid.to_numpy().sum() == len(topic_df)
    print(f"Heatmap dimensions: {len(topic_levels)} topic groups ? {len(breadth_levels)} breadth groups")
    print("Counts <30 are marked with *; this is a descriptive sparse-cell flag, not a precision guarantee.")
    display(joint_summary.round(6))

    def format_heatmap(ax):
        ax.set_xticks(np.arange(len(breadth_levels)))
        ax.set_xticklabels([str(v) for v in breadth_levels], rotation=35, ha="right")
        ax.set_yticks(np.arange(len(topic_levels)))
        ax.set_yticklabels([str(v) for v in topic_levels])
        ax.set_xlabel("Market breadth: markets_per_day quantile interval")
        ax.set_ylabel("TOPIC concentration proxy: largest transformer-topic share interval")

    values = median_grid.to_numpy(dtype=float)
    color_limit = max(float(np.nanmax(np.abs(values))), 1e-12)
    fig, ax = plt.subplots(figsize=(12, 8))
    cmap = plt.get_cmap("RdBu_r").copy()
    cmap.set_bad("lightgray")
    im = ax.imshow(np.ma.masked_invalid(values), origin="lower", aspect="auto", cmap=cmap, vmin=-color_limit, vmax=color_limit)
    for row, column in np.ndindex(values.shape):
        value = values[row, column]
        label = "No data" if not np.isfinite(value) else f"{value:.5f}" + ("*" if count_grid.iloc[row, column] < 30 else "")
        color = "white" if np.isfinite(value) and abs(value) > 0.6 * color_limit else "black"
        ax.text(column, row, label, ha="center", va="center", color=color, fontsize=9)
    format_heatmap(ax)
    ax.set_title("Median trader_ppv: market breadth ? TOPIC concentration proxy")
    fig.colorbar(im, ax=ax, label="Median profit per unit of volume (trader_ppv)")
    fig.tight_layout()
    plt.show()
else:
    print("Profit heatmap unavailable: no validated topic proxy or exact market concentration.")

In [ ]:
if topic_proxy_available:
    counts = count_grid.to_numpy()
    fig, ax = plt.subplots(figsize=(12, 8))
    im = ax.imshow(counts, origin="lower", aspect="auto", cmap="Blues", vmin=0)
    for row, column in np.ndindex(counts.shape):
        count = counts[row, column]
        color = "white" if count > 0.6 * counts.max() else "black"
        ax.text(column, row, f"{count:,}" + ("*" if 0 < count < 30 else ""), ha="center", va="center", color=color)
    format_heatmap(ax)
    ax.set_title("Trader counts: market breadth ? TOPIC concentration proxy")
    fig.colorbar(im, ax=ax, label="Number of traders")
    fig.tight_layout()
    plt.show()
else:
    print("Count heatmap unavailable for the same reason as the profit heatmap.")

# 6. Statistical Analysis

Calculate Pearson (linear) and Spearman (rank) correlations on untransformed trader-level values. Report PPV and P&L separately for market breadth and the available topic proxy. These are different constructs and may use different sample sizes. P-values are nominal, unadjusted for multiple comparisons, and assume independent observations; unique trader IDs do not imply independent ownership. Extreme finite observations can affect Pearson correlations and group means.

As a descriptive joint check, report topic-proxy correlations within market-breadth quintiles. This explores whether topic dominance has different associations at similar breadth; it still cannot identify market-level concentration.

In [ ]:
def correlations(data, predictor, outcome):
    x = data[predictor].to_numpy(dtype=float)
    y = data[outcome].to_numpy(dtype=float)
    result = {"n_traders": len(data)}
    for name, function in [("pearson", stats.pearsonr), ("spearman", stats.spearmanr)]:
        if len(data) < 3 or np.unique(x).size < 2 or np.unique(y).size < 2:
            coefficient, p_value = np.nan, np.nan
        else:
            coefficient, p_value = function(x, y)
        result[f"{name}_r"] = coefficient
        result[f"{name}_p"] = p_value
    return result

metrics = [("Market breadth (daily rate)", breadth_column, analysis_df)]
if topic_proxy_available:
    metrics.append(("TOPIC concentration proxy (not markets)", topic_proxy_column, topic_df))
correlation_table = pd.DataFrame([
    {"construct": construct, "metric": metric, "outcome": outcome, **correlations(data, metric, outcome)}
    for construct, metric, data in metrics
    for outcome in ["trader_ppv", "trader_pnl"]
])
formats = {"pearson_r": "{:.5f}", "spearman_r": "{:.5f}", "pearson_p": "{:.5g}", "spearman_p": "{:.5g}", "n_traders": "{:,.0f}"}
display(correlation_table.style.format(formats))
if topic_proxy_available:
    within_breadth_correlations = pd.DataFrame([
        {"breadth_group": str(group), "outcome": outcome, **correlations(sample, topic_proxy_column, outcome)}
        for group, sample in topic_df.groupby("heat_breadth_group", observed=True)
        for outcome in ["trader_ppv", "trader_pnl"]
    ])
    display(within_breadth_correlations.style.format(formats))

# 7. Key Findings

Findings below are computed from this run. Inspect effect sizes, means versus medians, cell counts, and secondary P&L results together. Grouped patterns do not control for tenure, transaction frequency, volume, strategy, or risk. Sparse-cell medians can be unstable; broad quantile intervals do not make traders identical.

In [ ]:
findings = [
    f"**Sample:** {len(analysis_df):,} unique traders with at least 100 transactions; {len(breadth_summary)} breadth groups.",
    "**Market-data limitation:** markets_per_day is available, but total distinct markets, per-market allocation, and effective market counts are not. The exact question about many markets with concentrated versus even participation remains unanswered.",
]
for _, row in correlation_table.iterrows():
    findings.append(
        f"**{row['metric']} vs {row['outcome']}:** Pearson r={row['pearson_r']:.4f} "
        f"(p={row['pearson_p']:.3g}); Spearman rho={row['spearman_r']:.4f} "
        f"(p={row['spearman_p']:.3g}); n={row['n_traders']:,}."
    )
first, last = breadth_summary.iloc[0], breadth_summary.iloc[-1]
findings.append(f"**Lowest to highest breadth group:** mean PPV {first['mean_trader_ppv']:.6f} ? {last['mean_trader_ppv']:.6f}; median PPV {first['median_trader_ppv']:.6f} ? {last['median_trader_ppv']:.6f}.")
if topic_proxy_available:
    findings.append(f"**Topic heatmaps:** {len(topic_df):,} traders; {int(((count_grid > 0) & (count_grid < 30)).to_numpy().sum())} nonempty cells have fewer than 30 traders. These display topic dominance, not concentration across individual markets.")
    findings.append("**Topic validation:** the supplied topic representation is numerically normalized and its maximum agrees with the existing largest-topic proxy on comparable rows. Overlap and weighting semantics remain undocumented; no effective topic count or market count was calculated.")
else:
    findings.append("**Topic heatmaps omitted:** proxy validation was unavailable or failed.")
findings.append("**Next data needed:** trader-to-market transaction allocations, stable market identifiers, and a common observation window. These would enable distinct market counts and inverse-squared-share effective market counts.")
findings.append("**Interpretation:** associations do not establish that broader trading or greater topic diversity causes higher profitability.")
display(Markdown("\n".join("- " + item for item in findings)))